# 💬 Conversational RAG with Chat History

Build a question-answering RAG system over a Wikipedia subset, watch it fail on a follow-up question, then make it
conversational: rewrite each question with the chat history before retrieving, and answer with that history in the prompt.

**Reference:** [RAG Systems Essentials (Analytics Vidhya)](https://courses.analyticsvidhya.com/courses/take/rag-systems-essentials/lessons/60148017-hands-on-deep-dive-into-rag-evaluation-metrics-generator-metrics-i)

## Learning Objectives
In this notebook, you will learn:
1. **Build a knowledge base** - download Wikipedia, keep a small subset, chunk it and index it in Chroma
2. **Basic RAG with LCEL** - a single-turn question-answering chain, and why it fails on follow-up questions
3. **Contextualize the question** - a history-aware retriever that rewrites follow-ups into standalone questions
4. **Conversational RAG chain** - `create_retrieval_chain` with chat history passed in on every turn

## Where this fits
`02_rag_system_for_question_answering.ipynb` answered one question at a time. This notebook adds conversation memory
for one user; `04_multi_user_conversational_rag_system.ipynb` then keeps a separate history for every user.

## Prerequisites
- `02_rag_system_for_question_answering.ipynb` (LCEL RAG chain) - background reading
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (OpenAI embeddings and `gpt-4o-mini`)
- Internet access: Part 1 downloads the Simple English Wikipedia dump from sbert.net on its first run

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, then create the chat model used for question rewriting and answers.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: gpt-4o-mini for question rewriting and answers
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0 is the factory default, as in the original lesson
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📚 Part 1: Build the Wikipedia Knowledge Base

Download the Simple English Wikipedia dump, keep the first paragraph of every article, and keep only passages that
mention *fish*, *india* or *cheetah* so that indexing stays fast and cheap. Then chunk the passages and index them in Chroma.

In [ ]:
# ============================================================================
# DOWNLOAD: Simple English Wikipedia dump (2020-11-01) from sbert.net
# ============================================================================
import gzip
import json
import requests
from tqdm import tqdm
import sys
import os

# Download a file from a URL
def http_get(url, path) -> None:
    """
    Downloads a URL to a given path on disc
    """
    if os.path.dirname(path) != "":
        os.makedirs(os.path.dirname(path), exist_ok=True)

    req = requests.get(url, stream=True)
    if req.status_code != 200:
        print("Exception when trying to download {}. Response {}".format(url, req.status_code), file=sys.stderr)
        req.raise_for_status()
        return

    download_filepath = path + "_part"
    with open(download_filepath, "wb") as file_binary:
        content_length = req.headers.get("Content-Length")
        total = int(content_length) if content_length is not None else None
        progress = tqdm(unit="B", total=total, unit_scale=True)
        for chunk in req.iter_content(chunk_size=1024):
            if chunk:  # filter out keep-alive new chunks
                progress.update(len(chunk))
                file_binary.write(chunk)

    os.rename(download_filepath, path)
    progress.close()


wikipedia_filepath = 'simplewiki-2020-11-01.jsonl.gz'

# download once; later runs (and the other conversational notebook) reuse the local file
if not os.path.exists(wikipedia_filepath):
    http_get('http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz', wikipedia_filepath)
print('✅ Wikipedia dump ready:', wikipedia_filepath)

In [ ]:
# ============================================================================
# PARSE: Keep the first paragraph of every article
# ============================================================================
import gzip
import json

wikipedia_filepath = 'simplewiki-2020-11-01.jsonl.gz'

passages = []
with gzip.open(wikipedia_filepath, 'rt', encoding='utf8') as fIn:
    for line in fIn:
        data = json.loads(line.strip())
        #Only add the first paragraph
        passages.append(data['paragraphs'][0])

print("Passages:", len(passages))

In [ ]:
# We subset our data so we only use a subset of wikipedia to run things faster
passages = [passage for passage in passages for x in ['fish', 'india', 'cheetah']
              if x in passage.lower().split()]

In [ ]:
len(passages)

In [ ]:
passages[0]

### Generate LLM Embeddings and store them in Chroma Vector DB

**Chroma Vector DB** is a versatile, open-source vector database designed for managing and querying vector embeddings. It is easy to set up and integrates well with various AI tools and algorithms. Chroma is particularly useful for applications that require rapid and precise retrieval of content represented as embeddings—efficient data formats for text, images, and soon, audio and video.

**Key Features:**
- **Integration with AI Tools:** Chroma supports embedding functions from leading providers like OpenAI, Google, and Hugging Face, allowing for flexible and powerful data handling.
- **Ease of Use:** The database provides default embedding functions, or users can integrate external APIs to generate embeddings.
- **Efficient Querying:** Users can create collections to store embeddings, documents, and metadata. These can be queried to retrieve the most similar items, making information retrieval quick and effective.
- **Flexible API:** Chroma offers a straightforward API that supports both standard operations and custom embedding functions.

For more detailed information, visit the official Chroma documentation [here](https://docs.trychroma.com).

In [ ]:
passages[:3]

In [ ]:
# ============================================================================
# EMBEDDINGS: OpenAI text-embedding-3-small
# ============================================================================
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
# The vectorstore we'll be using
from langchain_chroma import Chroma

# The splitting and chunking strategy
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_core.documents import Document

docs = [Document(page_content=doc) for doc in passages]

In [ ]:
docs[:3]

In [ ]:
# ============================================================================
# SPLIT: 3,000-character chunks with 200 characters of overlap
# ============================================================================
splitter = RecursiveCharacterTextSplitter(chunk_size=3000,
                                          chunk_overlap=200)
chunked_docs = splitter.split_documents(docs)

In [ ]:
chunked_docs[:3]

### Create Vector DB and Retriever

The next cell builds `./wiki_db` only when the folder does not exist yet: re-running `from_documents` on an existing
folder would store every chunk a second time. Delete the folder to rebuild it.

In [ ]:
# ============================================================================
# VECTOR DB: Embed the Wikipedia chunks into a persistent Chroma collection
# ============================================================================
from pathlib import Path

# create vector DB of docs and embeddings - takes about 1 min
# from_documents on an existing folder would store every document a second time,
# so the vector DB is built only once - delete ./wiki_db to rebuild it
if Path("./wiki_db").exists():
    print("✅ ./wiki_db already exists - the next cell loads it (delete the folder to rebuild)")
else:
    chroma_db = Chroma.from_documents(documents=chunked_docs, collection_name='wiki_db',
                                      embedding=openai_embed_model,
                                      # need to set the distance function to cosine else it uses euclidean by default
                                      # check https://docs.trychroma.com/guides#changing-the-distance-function
                                      collection_metadata={"hnsw:space": "cosine"},
                                      persist_directory="./wiki_db")
    print(f"✅ Indexed {len(chunked_docs)} documents into ./wiki_db")

### Load Vector DB from disk

This works whether the previous cell just built the DB or it already existed.

In [ ]:
# load from disk
chroma_db = Chroma(persist_directory="./wiki_db",
                   collection_name='wiki_db',
                   embedding_function=openai_embed_model)

In [ ]:
chroma_db

The retriever returns up to 5 chunks whose cosine similarity to the query is at least 0.2.

In [ ]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity_score_threshold",
                                              search_kwargs={"k": 5, "score_threshold": 0.2})

---
## 🔁 Part 2: Basic RAG Chain with LCEL

A single-turn chain: retrieve chunks for the question, put them in the prompt with the question, and let the LLM answer.

In [ ]:
# ============================================================================
# RAG PROMPT: Answer only from the retrieved context
# ============================================================================
from langchain_core.prompts import ChatPromptTemplate

prompt = """You are an assistant for question-answering tasks.
            Use the following pieces of retrieved context to answer the question.
            If the answer is not present in the context, just say that you don't know.
            Keep the answer to the point.

            Question:
            {question}

            Context:
            {context}

            Answer:
         """

prompt_template = ChatPromptTemplate.from_template(prompt)

In [ ]:
prompt_template.pretty_print()

In [ ]:
# ============================================================================
# RAG CHAIN: retriever -> format_docs -> prompt -> LLM
# ============================================================================
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

qa_rag_chain = (
    {
        "context": (similarity_retriever
                      |
                    format_docs),
        "question": RunnablePassthrough()
    }
      |
    prompt_template
      |
    llm
)

In [ ]:
query = "What is the fastest animal?"
result = qa_rag_chain.invoke(query)
print(result.content)

In [ ]:
query = "Who was the winner of the champions league in 2020?"
result = qa_rag_chain.invoke(query)
print(result.content)

In [ ]:
query = "What is the capital of India?"
result = qa_rag_chain.invoke(query)
print(result.content)

In [ ]:
query = "Tell me more about it in detail"
result = qa_rag_chain.invoke(query)
print(result.content)

Read the last two answers together. "Tell me more about it in detail" only makes sense after the previous
question, but this chain sees one question at a time: the retriever searches for the literal words, and the LLM cannot
know what "it" refers to. The Champions League question tests the "say you don't know" instruction, because the
fish/India/cheetah subset has no football articles. Parts 3 and 4 fix the follow-up problem.

---
## 🧭 Part 3: Contextualize the Question

In many Q&A applications, the ability to engage in back-and-forth conversations with users is crucial. This necessitates the application having a form of "memory" to recall past interactions and apply this context to current queries.

This guide focuses on integrating historical messages into the application's logic. Additional details on managing chat history can be found [here](https://python.langchain.com/docs/expression_language/how_to/message_history/).

![](https://i.imgur.com/8hLJMPl.gif)

### Building on the Q&A RAG System - to a Conversational Q&A RAG System

We will enhance our Q&A RAG System, which utilizes the Wikipedia dataset, by implementing the following updates:

- **Prompt Adjustment:** Our prompt will be modified to include historical messages as inputs, allowing the system to maintain context over the course of a conversation.

- **Contextualizing Questions:** We will introduce a sub-chain mechanism to reformulate the latest user query by considering the chat history. This is crucial for understanding questions that refer back to previous messages. For example, a query like "Can you elaborate on the second point?" relies on the context provided by preceding interactions, which affects the system's ability to retrieve relevant information effectively.

## Contextualizing the Question

To maintain a seamless flow in conversations, especially in a Q&A setting, it's essential to incorporate historical interactions. Here’s how we achieve this:

### Defining a Sub-Chain for Historical Context

1. **Sub-Chain Creation:** We'll define a sub-chain that uses both historical messages and the latest user query. This sub-chain reformulates the question if it refers to any past interactions, ensuring the system, especially the vector database understands the context to return the most relevant documents to this newly reworded question.

2. **Using `MessagesPlaceholder`:** Our prompt construction involves a `MessagesPlaceholder` variable named `chat_history`. This setup allows us to input a list of messages using the `chat_history` key. The system integrates these messages, positioning them after its own responses and before the latest user question.

3. **Helper Function Usage:** We employ the `create_history_aware_retriever` function available [here](https://api.python.langchain.com/en/latest/chains/langchain.chains.history_aware_retriever.create_history_aware_retriever.html). This function is crucial for handling instances where the chat history might be empty and orchestrates the sequence of operations: `prompt | llm | StrOutputParser() | retriever`.

4. **Chain Construction:** The `create_history_aware_retriever` constructs a chain that processes inputs under the keys `input` and `chat_history`, ensuring the output schema aligns with that of a retriever.

By implementing these steps, our system can effectively utilize historical context to better understand and respond to user queries, thereby enhancing the conversational experience.

In [ ]:
# ============================================================================
# HISTORY-AWARE RETRIEVER: Rewrite follow-ups into standalone questions, then retrieve
# ============================================================================
from langchain_classic.chains import create_history_aware_retriever
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

rephrase_system_prompt = """Given a chat history and the latest user question
which might reference context in the chat history, formulate a standalone question
which can be understood without the chat history. Do NOT answer the question,
just reformulate it if needed and otherwise return it as is.
"""

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm, similarity_retriever, rephrase_prompt
)

This chain prepends a rephrasing of the input query to our retriever, so that the retrieval incorporates the context of the conversation.

---
## 💬 Part 4: Conversational RAG Chain

Now we're ready to construct our comprehensive QA RAG chain, which leverages historical context for more accurate and relevant responses.

### Components of the QA RAG Chain

1. **Creating Document Chains:**
   - We use the `create_stuff_documents_chain` function, which is detailed [here](https://api.python.langchain.com/en/latest/chains/langchain.chains.combine_documents.stuff.create_stuff_documents_chain.html). This function is used to create a `question_answer_chain`, accepting inputs such as `context`, `chat_history`, and `input`. It efficiently combines the retrieved context with the conversation history and the current query to generate an informed answer.

2. **Building the Final QA RAG Chain:**
   - The entire QA RAG chain is assembled using the `create_retrieval_chain` function, available [here](https://api.python.langchain.com/en/latest/chains/langchain.chains.retrieval.create_retrieval_chain.html). This chain integrates the `history_aware_retriever` with the `question_answer_chain`. It retains intermediate outputs like the retrieved context for added convenience during the query handling process.
   - The `create_retrieval_chain` function accepts keys such as `input` and `chat_history` and includes `input`, `chat_history`, `context`, and `answer` in its outputs.

By implementing these steps, the system not only contextualizes but also provides accurate answers by synthesizing information from both the current and historical interactions. This method enhances the conversational AI’s ability to understand and respond to user queries dynamically, making the interactions more engaging and relevant.

In [ ]:
# ============================================================================
# CONVERSATIONAL RAG CHAIN: history-aware retriever + stuff-documents QA chain
# ============================================================================
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

qa_system_prompt = """You are an assistant for question-answering tasks.
                      Use the following pieces of retrieved context to answer the question.
                      If the answer is not present in the context, just say that you don't know.
                      Keep the answer to the point.

                      Context:
                      {context}

                  """

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", """Question:
                     {input}

                     Answer:
                  """),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
qa_rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

In [ ]:
chat_history = []

question = "What is the capital of India?"
response = qa_rag_chain.invoke({"input": question, "chat_history": chat_history})
print(response['answer'])

In [ ]:
chat_history

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history.extend([HumanMessage(content=question),
                     AIMessage(content=response["answer"])])
chat_history

In [ ]:
question = "Tell me more about this city"
response = qa_rag_chain.invoke({"input": question, "chat_history": chat_history})
print(response['answer'])

In [ ]:
chat_history.extend([HumanMessage(content=question),
                     AIMessage(content=response["answer"])])
chat_history

In [ ]:
question = "What is the fastest animal?"
response = qa_rag_chain.invoke({"input": question, "chat_history": chat_history})
print(response['answer'])

In [ ]:
response

In [ ]:
chat_history.extend([HumanMessage(content=question),
                     AIMessage(content=response["answer"])])

In [ ]:
chat_history

In [ ]:
chat_history[-2:]

In [ ]:
question = "Tell me about its different species"
response = qa_rag_chain.invoke({"input": question, "chat_history": chat_history})
chat_history.extend([HumanMessage(content=question),
                     AIMessage(content=response["answer"])])
print(response['answer'])

Compare these follow-ups with Part 2. "Tell me more about this city" and "Tell me about its different species"
reach the retriever as rewritten, standalone questions, so they retrieve passages about the city and the animal from the
previous answers. Here the chat history is a plain Python list that we extend after every turn - fine for one user,
but every user of a real chatbot needs their own history.

---
## 📝 Summary

In this notebook, we learned:

### 1. Build a Knowledge Base
- A small, filtered Wikipedia subset keeps indexing fast; the Chroma folder is built once and reloaded afterwards.

### 2. Basic RAG with LCEL
- A single-turn RAG chain answers standalone questions but cannot resolve "it" or "this city" in a follow-up.

### 3. Contextualize the Question
- `create_history_aware_retriever` runs `prompt | llm | StrOutputParser() | retriever`, so the retriever searches with
  a standalone rewrite of the follow-up question.

### 4. Conversational RAG Chain
- `create_retrieval_chain` returns `input`, `chat_history`, `context` and `answer`; the caller appends every turn to
  the history.

### Next Steps
- **`04_multi_user_conversational_rag_system.ipynb`**: stores a separate, SQL-backed history per user session and adds
  it automatically with `RunnableWithMessageHistory`.